# 02 — DeepSeek-MoE Architecture Lab

Prototype and explore the **DeepSeek-MoE** architecture using the building blocks in
`src/core/`.  This is where you iterate before promoting to `src/`.

In [1]:
import sys
sys.path.insert(0, '../..')

import torch
from src.models.deepseek_moe.config import DeepSeekMoEConfig
from src.models.deepseek_moe.model import DeepSeekMoESLM
from src.core.generation import generate
from src.utils.training import count_params

In [2]:
# ── Build a tiny model for fast iteration ────────────────────────────────
tiny_cfg = DeepSeekMoEConfig(
    vocab_size=50257, block_size=32,
    n_layer=2, n_head=2, n_kv_head=1, n_embd=64,
    intermediate_size=128,
    n_shared_experts=1, n_routed_experts=4, top_k=2,
    expert_hidden_dim=64, dense_layers=[0],
    dropout=0.0,
)
model = DeepSeekMoESLM(tiny_cfg)
print(f'Parameters: {count_params(model):,}')

Parameters: 3,327,616


In [3]:
# ── Verify forward pass shapes ────────────────────────────────────────────
B, T = 2, tiny_cfg.block_size
idx = torch.randint(0, tiny_cfg.vocab_size, (B, T))
targets = torch.randint(0, tiny_cfg.vocab_size, (B, T))

logits, loss = model(idx, targets)
print(f'logits: {logits.shape}  |  loss: {loss.item():.4f}')

logits: torch.Size([2, 32, 50257])  |  loss: 10.8308


In [4]:
# ── Test generation ───────────────────────────────────────────────────────
prompt = torch.randint(0, tiny_cfg.vocab_size, (1, 4))
output = generate(model, prompt, max_new_tokens=10, temperature=1.0, top_k=50)
print(f'Input tokens:  {prompt[0].tolist()}')
print(f'Output tokens: {output[0].tolist()}')

Input tokens:  [30358, 6075, 48931, 14811]
Output tokens: [30358, 6075, 48931, 14811, 17163, 31685, 19554, 19554, 8589, 42407, 46864, 46864, 44609, 27342]


In [5]:
# ── Parameter count comparison across model sizes ────────────────────────
configs = {
    'deepseek_tiny':   DeepSeekMoEConfig(vocab_size=50257, block_size=16,  n_layer=2,  n_head=2,  n_kv_head=1, n_embd=64,  intermediate_size=128,  expert_hidden_dim=64,  dense_layers=[0]),
    'deepseek_small':  DeepSeekMoEConfig(vocab_size=50257, block_size=128, n_layer=6,  n_head=6,  n_kv_head=2, n_embd=384, intermediate_size=1024, expert_hidden_dim=256, dense_layers=[0, 1]),
    'deepseek_medium': DeepSeekMoEConfig(vocab_size=50257, block_size=256, n_layer=12, n_head=12, n_kv_head=4, n_embd=768, intermediate_size=2048, expert_hidden_dim=512, dense_layers=[0, 1]),
}
for name, cfg in configs.items():
    n = count_params(DeepSeekMoESLM(cfg))
    print(f'{name:16s}: {n:>12,} params  ({n/1e6:.1f}M)')

deepseek_tiny   :    3,377,024 params  (3.4M)
deepseek_small  :   34,651,392 params  (34.7M)
deepseek_medium :  173,157,888 params  (173.2M)
